In [9]:
import cv2
import pytesseract
import json
import mysql.connector
import re


def connect_database():
    try:
        connection = mysql.connector.connect(
            host="localhost",
            user="root",
            password="",
            database="patient_db"
        )
        print("Connected to the database")
        return connection
    except mysql.connector.Error as err:
        print(f"Database connection failed: {err}")
        return None


def preprocess_image(image_path):
    image = cv2.imread(image_path)
   
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    binary_image = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    
    cv2.imwrite('preprocessed_image.jpg', binary_image)
    return binary_image

def extract_text(image):
    custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist="0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ:-"'
    text = pytesseract.image_to_string(image, config=custom_config)
    print("Raw OCR Output:")
    print(text)
    return text

def parse_extracted_text(text):
    
    patient_name_match = re.search(r'Patient Name\s*:?\s*([A-Za-z ]+)', text)
    dob_match = re.search(r'DOB\s*:?\s*([0-9]{2}/[0-9]{2}/[0-9]{4})', text)
    pain_match = re.search(r'Pain\s*:?\s*(\d+)', text)
    numbness_match = re.search(r'Numbness\s*:?\s*(\d+)', text)
    tingling_match = re.search(r'Tingling\s*:?\s*(\d+)', text)

   
    patient_name = patient_name_match.group(1).strip() if patient_name_match else "Unknown"
    dob = dob_match.group(1).strip() if dob_match else "Unknown"
    pain = int(pain_match.group(1)) if pain_match else 0
    numbness = int(numbness_match.group(1)) if numbness_match else 0
    tingling = int(tingling_match.group(1)) if tingling_match else 0

   
    pain = min(max(pain, 0), 10)
    numbness = min(max(numbness, 0), 10)
    tingling = min(max(tingling, 0), 10)

    return {
        "patient_name": patient_name,
        "dob": dob,
        "pain": pain,
        "numbness": numbness,
        "tingling": tingling
    }


def insert_data_into_database(connection, data):
    try:
        cursor = connection.cursor()
        query = (
            "INSERT INTO patients (name, dob, pain, numbness, tingling) "
            "VALUES (%s, %s, %s, %s, %s)"
        )
        
        dob_value = data['dob'] if data['dob'] != "Unknown" else None
        values = (data['patient_name'], dob_value , data['pain'], data['numbness'], data['tingling'])
        cursor.execute(query, values)
        connection.commit()
        print("Data successfully inserted into the database.")
    except mysql.connector.Error as err:
        print(f"Data insertion failed: {err}")
    finally:
        cursor.close()


def main():
   
    connection = connect_database()
    if not connection:
        return

    
    image_path = 'newform.jpg'
    preprocessed_image = preprocess_image(image_path)

   
    extracted_text = extract_text(preprocessed_image)

    
    extracted_data = parse_extracted_text(extracted_text)

    
    print("Extracted JSON Data:")
    print(json.dumps(extracted_data, indent=4))

    
    with open("extracted_data.txt", "w") as file:
        json.dump(extracted_data, file, indent=4)
    
    print("Extracted JSON data has been saved to 'extracted_data.txt'.")

    insert_data_into_database(connection, extracted_data)

   
    connection.close()

if __name__ == "__main__":
    main()


Connected to the database
Raw OCR Output:
FundtionalAssessmentQuestionnire
BondingorStoopings022345" :
Puttingonshoes012345
cingupordownafghtofstai012345
ringtoanher042345
Eeppagenmanon2a4s:
Bescrite-anyfunctionalchangeswithinthelastthreedaysgoodarbad
WiopecontictedtyMsa ems

Extracted JSON Data:
{
    "patient_name": "Unknown",
    "dob": "Unknown",
    "pain": 0,
    "numbness": 0,
    "tingling": 0
}
Extracted JSON data has been saved to 'extracted_data.txt'.
Data successfully inserted into the database.
